In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import os
import datetime
import matplotlib.dates as mdates

pd.plotting.register_matplotlib_converters()

In [2]:
# Store site metadata (site number, site name, catchment number, NZTME, NZTMN)

metadata_df = pd.read_csv('/nesi/project/niwa00015/queenle/data/metadata/40_50yr_sites.csv')
metadata_df.set_index('Number',inplace=True)

# define alpha and numeric season order
seasons_a = ['fall','winter','spring','summer']
seasons_n = [3,6,9,12]

In [3]:
'''
Helper function for null value handling. 
Returns the maximum length of consecutive nan values in an array.
'''

def max_consecutive_nans(a):
    mask = np.concatenate(([False],np.isnan(a),[False]))
    if ~mask.any():
        return 0
    else:
        idx = np.nonzero(mask[1:] != mask[:-1])[0]
        return (idx[1::2] - idx[::2]).max()

In [4]:
'''
Resample daily flow to seasonal based on the following missing value criteria:
- if season has more than 1/3 missing days = nan season mean
- if season has more than 7 consecutive missing days = nan season mean
'''

def day_to_season(df):
    
    start_year = int(df.index.year[0])
    end_year = int(df.index.year[-1])
    end_month = int(df.index.month[-1])
    start_month = int(df.index.month[0] )
    
    df_dict = {'date':[], 'flow':[]}

    for year in range(start_year, end_year + 1):
        for month in range(1,13):
            if (((year == start_year) and (month < start_month)) or ((year == end_year) and (month > end_month))):
                continue
            
            if month in [12,3,6,9]:
                dt = datetime.date(year, month, 1)
                
                if month == 12:
                    if year + 1 > end_year:
                        break
                    year_data = df[str(year)]
                    dec = year_data[year_data.index.month == 12]#.flow
                    next_year_data = df[str(year + 1)]
                    jan_feb = next_year_data[next_year_data.index.month.isin([1,2])]#.flow
                    season_data = pd.concat([dec, jan_feb])
                else:    
                    year_data = df[str(year)]
                    season_data = year_data[year_data.index.month.isin([month, month+1, month+2])]#.flow

                percentage_null = float(season_data.isnull().sum() / len(season_data))
                                
                df_dict['date'].append(dt)
                if (percentage_null > .33) or (max_consecutive_nans(np.array(season_data.values.tolist()).flatten()) > 7):
                    df_dict['flow'].append(math.nan)

                else:
                    df_dict['flow'].append(np.nanmean(season_data.values))
                    
    new_df = pd.DataFrame(data=df_dict)
    new_df.date = pd.to_datetime(new_df.date)
    new_df.set_index('date', inplace=True)
    return new_df

In [5]:
'''
resampling data from daily to seasonal, writing to new csv files
'''

filepath = '/nesi/project/niwa00015/queenle/data/flows/daily/40plus/'

df_list = []

for filename in os.listdir(filepath):
    
    if filename == 'full_records.csv':
        continue

    header = open(filepath + filename).readline().strip().split(' ')
    site_number = header[1]
    site_name = " ".join(header[2:])
    
    site_df = pd.read_csv(filepath + filename, header=1)
    site_df.Date = pd.to_datetime(site_df.Date)
    site_df.set_index('Date', inplace=True)
    site_df.columns = ['time','flow']
    site_df = site_df.drop(columns='time')
    site_df = site_df.replace('gap ', math.nan)
    site_df["flow"] = pd.to_numeric(site_df["flow"])
            
    replace_dates = {}
    for date in site_df.index:
        if date.year > 2020:
            new_date = date.replace(year=date.year - 100)
            replace_dates[date] = new_date
            
    site_df.rename(replace_dates,inplace=True)
        
    data = day_to_season(site_df)
    
    data.columns = [site_number]
    
    df_list.append(data)
    
    #data.to_csv("/nesi/project/niwa00015/queenle/data/flows/seasonal/50yrs/individual_sites/" + site_number + ".csv")

combined_df = pd.concat(df_list,axis=1)

    

In [1]:
combined_df.to_csv("/nesi/project/niwa00015/queenle/data/flows/seasonal/40_50yrs/40plus.csv")

NameError: name 'combined_df' is not defined